# 4-Way Architecture Screening: Alternatives to the ResNet Block

**Honest scope statement — read this before interpreting any result below.**

This is a **fast screening study**, not a rigorous reproduction of any cited paper. All 4
candidates are **adapted** to share the exact same input/output interface as the existing
pipeline (64×64×2 in → 256×256×1 heatmap out, same blob-detector evaluator) so they can be
compared fairly and cheaply — only the **core processing block** changes, replacing the
64-stacked-residual-block body.

Compromises made for time/scope, stated explicitly:
- **8 blocks each** (not 64) — none of these have pretrained weights to warm-start from
  (unlike the ResNet compression project), so all 4 train from scratch; a shallower depth
  keeps this a fair, fast, equal-footing comparison, not a claim about paper-scale performance.
- Each gets an **identical, hard-capped training budget** (~32.5 min out of a 2.5h total,
  automatically computed from `TOTAL_TIME_BUDGET_HOURS` minus a protected eval reserve) —
  enough to see whether an architecture learns at all and how its loss trends, not enough to
  fully converge.
- Each is a **simplified adaptation** of its literature family, not the literal cited method
  (e.g. the "graph/message-passing" candidate is implemented as local-window self-attention
  over the spatial grid, not a true antenna-element graph, which would need a different
  input representation entirely).
- **The eval reserve is measured, not guessed.** `evaluate_on_bank`'s cost is dominated by
  per-sample CPU blob detection + Hungarian matching, not GPU inference, so it doesn't scale
  the way training time does. Cell 12b runs a small real benchmark on all 4 architectures
  *and* the teacher (all 5 models Cell 15 actually evaluates) before training starts, and
  auto-shrinks `N_PER_SNR_EVAL` if the measured throughput would blow through
  `EVAL_RESERVE_MINUTES`. If you see a "MEASURED throughput will NOT fit" message, the eval
  set was made smaller automatically — check Cell 12b's printed output for the actual numbers
  used in the final comparison.
- **Fixed wall-clock budget, not fixed step count**, means architectures with cheaper
  per-step cost (SIREN, Gridless-Unfold: plain convs) get more gradient updates than the
  heavier ones (FNO: FFT/iFFT per block; Window-Attention: reshape + multi-head attention
  per block) in the same training window. This is an accepted screening-study trade-off
  (equal *time*, not equal *updates*) — a promising candidate here deserves a
  step-count-matched follow-up, not a paper-scale claim from this run alone.

**What this notebook answers:** which of these 4 families shows enough promise, this cheaply,
to justify a real (deeper, longer, more faithful) investment later. **What it does NOT
answer:** which one is definitively best at paper scale.

### The 4 candidates (see prior research discussion for full literature grounding)
1. **SIREN-body** — sin() activations throughout, targets spectral bias (our signal *is* literally a sum of sinusoids)
2. **FNO-body** — spectral (Fourier-domain) convolution, global receptive field, matches the DFT-structured physics
3. **Window-Attention-body** — local neighborhood self-attention (message-passing/graph-style local aggregation, adapted to a fixed image grid)
4. **Gridless-Unfold-body** — learned complex soft-thresholding refinement (deep-unfolding style), explicitly fixing the complex-phase bug PIA-Net had (ReLU-clamped real values can't represent an arbitrary-phase path; this uses magnitude-soft-threshold + exact phase preservation instead — verified locally before writing this notebook)

### Kaggle attach
- `dldoa-source-code`, `dldoa-frozen-banks` (same as the compression project)

In [ ]:
# Cell 1 — Setup + wall-clock start
import importlib, subprocess, sys, time
T_START = time.time()

try:
    import cv2
except ImportError:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'opencv-python-headless'], check=True)

import os, json
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras.layers import Conv2D, Input, BatchNormalization, Activation, Add, Conv2DTranspose, Layer
from tensorflow.keras.models import Model

tf.keras.backend.clear_session()
tf.get_logger().setLevel('ERROR')
np.random.seed(42); tf.random.set_seed(42)

gpus = tf.config.list_physical_devices('GPU')
print(f'TF: {tf.__version__}  |  GPU: {gpus}')
for g in gpus: tf.config.experimental.set_memory_growth(g, True)

OUT_DIR = '/kaggle/working'
os.makedirs(OUT_DIR, exist_ok=True)

def elapsed_hours():
    return (time.time() - T_START) / 3600

print('Wall clock started. t=0.00h')

In [ ]:
# Cell 2 — Locate repo source, teacher weights, frozen eval bank, training generator
def find_path(name_pattern):
    from pathlib import Path
    for root in ['/kaggle/input', '/kaggle/working', '.', '/content']:
        if not os.path.isdir(root): continue
        for p in Path(root).rglob(name_pattern):
            if p.is_file(): return str(p)
    return None

SRC_MODEL_FILE = find_path('tvt_models.py')
assert SRC_MODEL_FILE is not None, 'DL_DOA source not found'
DL_DOA_DIR = os.path.dirname(os.path.dirname(SRC_MODEL_FILE))
print(f'DL_DOA dir: {DL_DOA_DIR}')

TEACHER_WEIGHTS_PATH = find_path('inf_model_007_256_resnet.h5')
EVAL_BANK_PATH = find_path('eval_bank.npz')
assert EVAL_BANK_PATH is not None, 'eval_bank.npz not found -- attach dldoa-frozen-banks'
TRAIN_GEN_FILE = find_path('dldoa_dataset_generation.py')
assert TRAIN_GEN_FILE is not None, 'dldoa_dataset_generation.py not found'
print(f'Teacher weights: {TEACHER_WEIGHTS_PATH}')
print(f'Eval bank: {EVAL_BANK_PATH}')
print(f'Training generator: {TRAIN_GEN_FILE}')

In [ ]:
# Cell 3 — Import ORIGINAL evaluator + Resnet + training generator
sys.path.insert(0, DL_DOA_DIR)
sys.path.insert(0, os.path.dirname(TRAIN_GEN_FILE))
from src.tvt_models import Resnet
from src.TVT_Blob_Inference import get_blob_detector, get_blob_peaks, peaks_to_angles, prepare_for_metric, get_ang_difference, filter_angles
from dldoa_dataset_generation import training_data_generator
print('✅ Imported')

In [ ]:
# Cell 4 — Load frozen eval bank + teacher (reference baseline only, never modified)
eval_bank = np.load(EVAL_BANK_PATH)
EVAL_DATA, EVAL_FEAT, EVAL_META = eval_bank['data'], eval_bank['feat'], eval_bank['meta']
SIGMA = float(eval_bank['sigma']); M = int(eval_bank['M'])

teacher = None
if TEACHER_WEIGHTS_PATH:
    teacher = Resnet(input_shape=(64, 64, 2))
    teacher.load_weights(TEACHER_WEIGHTS_PATH)
    teacher.trainable = False
    print(f'✅ Teacher loaded: {teacher.count_params():,} params (reference only)')
else:
    print('⚠️ Teacher weights not found -- proceeding without a reference row')

In [ ]:
N_BLOCKS = 8
FILTERS = 12
TOTAL_TIME_BUDGET_HOURS = 2.5
EVAL_RESERVE_MINUTES = 20
N_PER_SNR_EVAL = 100
STEPS_PER_EPOCH = 100

N_ARCHS = 4
PER_ARCH_BUDGET_HOURS = max(0.1, (TOTAL_TIME_BUDGET_HOURS - EVAL_RESERVE_MINUTES/60) / N_ARCHS)

print('N_BLOCKS=', N_BLOCKS)
print('Eval reserve (min):', EVAL_RESERVE_MINUTES)
print('Per-architecture training budget (min):', round(PER_ARCH_BUDGET_HOURS*60, 1))
print('Total training across', N_ARCHS, 'archs (min):', round(PER_ARCH_BUDGET_HOURS*60*N_ARCHS))
print('Total budget (min):', TOTAL_TIME_BUDGET_HOURS*60)

In [ ]:
# Cell 6 — Shared I/O wrapper + FNO's custom spectral-conv layer
# All 4 architectures plug a "body_fn(x, filters, n_blocks)" into this exact same shell,
# so only the core block differs -- everything else (input/output size, training loop,
# evaluator) is identical across all 4.

def build_model_with_body(body_fn, filters=FILTERS, n_blocks=N_BLOCKS, name='model'):
    x_in = Input(shape=(64, 64, 2))
    x = Conv2DTranspose(filters, (5, 5), strides=(2, 2), padding='same')(x_in)
    x = body_fn(x, filters, n_blocks)
    x = Conv2DTranspose(1, (5, 5), strides=(2, 2), padding='same')(x)
    return Model(x_in, x, name=name)

class SpectralConv2D(Layer):
    """FNO-style spectral convolution: rfft2 -> truncate to `modes` low frequencies ->
    learned COMPLEX channel-mixing weight -> pad back -> irfft2. Verified locally
    (pure numpy) before writing this: finite output, correct shape, sane scale.
    Complex weight = two real (trainable) tensors combined via tf.complex at call time,
    which keeps gradients well-defined in TF."""
    def __init__(self, out_channels, modes=12, **kwargs):
        super().__init__(**kwargs)
        self.out_channels = out_channels
        self.modes = modes

    def build(self, input_shape):
        in_ch = input_shape[-1]
        self.w_real = self.add_weight(shape=(self.modes, self.modes, in_ch, self.out_channels),
                                      initializer='glorot_uniform', trainable=True, name='w_real')
        self.w_imag = self.add_weight(shape=(self.modes, self.modes, in_ch, self.out_channels),
                                      initializer='glorot_uniform', trainable=True, name='w_imag')

    def call(self, x):
        H, W = x.shape[1], x.shape[2]
        x_chfirst = tf.transpose(x, [0, 3, 1, 2])                    # (B,C,H,W)
        x_ft = tf.signal.rfft2d(x_chfirst)                           # (B,C,H,W//2+1) complex
        x_ft = tf.transpose(x_ft, [0, 2, 3, 1])                      # (B,H,W//2+1,C)

        m1 = min(self.modes, H); m2 = min(self.modes, x_ft.shape[2])
        x_ft_trunc = x_ft[:, :m1, :m2, :]

        Wc = tf.complex(self.w_real[:m1, :m2], self.w_imag[:m1, :m2])
        out_ft_trunc = tf.einsum('bhwi,hwio->bhwo', x_ft_trunc, Wc)

        pad_h = H - m1; pad_w = (W // 2 + 1) - m2
        out_ft = tf.pad(out_ft_trunc, [[0, 0], [0, pad_h], [0, pad_w], [0, 0]])
        out_ft = tf.transpose(out_ft, [0, 3, 1, 2])
        out = tf.signal.irfft2d(out_ft, fft_length=[H, W])           # (B,C_out,H,W) real
        return tf.transpose(out, [0, 2, 3, 1])                       # (B,H,W,C_out)

print('✅ Shared shell + SpectralConv2D ready')

In [ ]:
# Cell 7 — Architecture A: SIREN-body (sin() activations, targets spectral bias)
def siren_conv(x, filters, is_first=False, omega=30.0):
    in_ch = x.shape[-1]
    limit = (1.0 / in_ch) if is_first else (np.sqrt(6.0 / in_ch) / omega)
    init = tf.keras.initializers.RandomUniform(-limit, limit)
    x = Conv2D(filters, 1, padding='same', kernel_initializer=init, bias_initializer=init)(x)
    return tf.sin(omega * x) if is_first else tf.sin(x)

def siren_block(x, filters):
    skip = x
    x = siren_conv(x, filters)
    x = siren_conv(x, filters)
    return Add()([x, skip])

def siren_body(x, filters, n_blocks):
    x = siren_conv(x, filters, is_first=True, omega=30.0)
    for _ in range(n_blocks):
        x = siren_block(x, filters)
    return x

print('✅ SIREN body ready')

In [ ]:
# Cell 8 — Architecture B: FNO-body (spectral convolution, global receptive field)
def fno_block(x, filters, modes=12):
    skip = x
    spec = SpectralConv2D(filters, modes=modes)(x)      # frequency-domain path
    local = Conv2D(filters, 1, padding='same')(x)         # parallel spatial path (standard FNO design)
    x = Add()([spec, local])
    x = BatchNormalization()(x); x = Activation('relu')(x)
    x = Add()([x, skip])
    return x

def fno_body(x, filters, n_blocks):
    for _ in range(n_blocks):
        x = fno_block(x, filters)
    return x

print('✅ FNO body ready')

In [ ]:
# Cell 9 — Architecture C: Window-Attention-body (local neighborhood self-attention,
# a message-passing/graph-style local aggregation adapted to a fixed image grid --
# NOT the literature's literal antenna-element graph, see the scope note above)
def window_attention_block(x, filters, window=8, num_heads=2):
    B = tf.shape(x)[0]; H = x.shape[1]; W = x.shape[2]; C = x.shape[-1]
    skip = x
    xw = tf.reshape(x, [B, H // window, window, W // window, window, C])
    xw = tf.transpose(xw, [0, 1, 3, 2, 4, 5])
    xw = tf.reshape(xw, [-1, window * window, C])

    mha = tf.keras.layers.MultiHeadAttention(num_heads=num_heads, key_dim=max(C // num_heads, 1))
    attn = mha(xw, xw)

    attn = tf.reshape(attn, [B, H // window, W // window, window, window, C])
    attn = tf.transpose(attn, [0, 1, 3, 2, 4, 5])
    attn = tf.reshape(attn, [B, H, W, C])

    x = BatchNormalization()(attn)
    x = Conv2D(filters, 1, padding='same')(x)             # channel-mixing after attention
    x = Add()([x, skip]); x = Activation('relu')(x)
    return x

def window_attention_body(x, filters, n_blocks):
    for _ in range(n_blocks):
        x = window_attention_block(x, filters)
    return x

print('✅ Window-attention body ready')

In [ ]:
# Cell 10 — Architecture D: Gridless-Unfold-body (learned complex soft-threshold refinement)
# Fixes PIA-Net's known bug: soft-threshold the MAGNITUDE, preserve the PHASE exactly,
# instead of ReLU-clamping to real-nonnegative (which cannot represent an arbitrary-phase
# path gain, e.g. a purely-imaginary value). Verified locally before writing this notebook.

def gridless_unfold_block(x, filters):
    skip = x
    z = Conv2D(filters, 5, padding='same')(x)              # linear "gradient step" (learned mixing)
    z = BatchNormalization()(z)

    re = z[..., 0::2]; im = z[..., 1::2]                    # even/odd channels as (real,imag) pairs
    mag = tf.sqrt(re**2 + im**2 + 1e-6)
    thresh = tf.Variable(0.1, trainable=True, dtype=tf.float32, name='soft_thresh')
    new_mag = tf.nn.relu(mag - tf.nn.softplus(thresh))      # softplus keeps threshold positive, still learnable
    scale = new_mag / (mag + 1e-6)
    re_out = re * scale; im_out = im * scale

    stacked = tf.stack([re_out, im_out], axis=-1)            # interleave back -- verified locally with numpy
    out = tf.reshape(stacked, tf.shape(z))

    out = Add()([out, skip]); out = Activation('relu')(out)
    return out

def gridless_unfold_body(x, filters, n_blocks):
    for _ in range(n_blocks):
        x = gridless_unfold_block(x, filters)
    return x

print('✅ Gridless-unfold body ready')

In [ ]:
# Cell 11 — Build all 4 models, report parameter counts
ARCHS = {
    'SIREN': siren_body,
    'FNO': fno_body,
    'WindowAttention': window_attention_body,
    'GridlessUnfold': gridless_unfold_body,
}

models = {}
for name, body_fn in ARCHS.items():
    m = build_model_with_body(body_fn, name=name)
    models[name] = m
    print(f'{name:>16}: {m.count_params():>10,} params')
if teacher is not None:
    print(f'{"Teacher (ref)":>16}: {teacher.count_params():>10,} params  (64 blocks, pretrained -- NOT an apples-to-apples depth comparison)')

In [ ]:
# Cell 12 — Evaluation function (reuses the ORIGINAL evaluator, same convention as every
# other notebook in this project)
def evaluate_on_bank(model, data_arr, feat_arr, meta_arr, batch_size=8, max_deg_error=1.0):
    detector = get_blob_detector()
    results_by_snr = {}
    N = data_arr.shape[0]
    for start in range(0, N, batch_size):
        end = min(start + batch_size, N)
        preds = model(data_arr[start:end], training=False)
        for j in range(end - start):
            idx = start + j
            L = int(meta_arr[idx, 0]); snr = int(meta_arr[idx, 1])
            peaks, amps = get_blob_peaks(preds[j], detector)
            order = np.argsort(-amps); peaks = peaks[order[:L]]
            angles_est = peaks_to_angles(peaks, sigma=SIGMA, grid_size=M)
            gt_angles, pred_angles = prepare_for_metric(angles_est, feat_arr[idx])
            results_by_snr.setdefault(snr, []).append((gt_angles, pred_angles))
    final_pd, final_rmse = {}, {}
    for snr, examples in results_by_snr.items():
        good_all, bad_all = [], []
        for gt, pred in examples:
            if np.isnan(pred).any(): continue
            diffs = get_ang_difference(gt, pred)
            good, bad = filter_angles(diffs, max_deg_error=max_deg_error)
            good_all.append(good); bad_all.append(bad)
        good_all = np.concatenate(good_all) if good_all else np.array([])
        bad_all = np.concatenate(bad_all) if bad_all else np.array([])
        total = len(good_all) + len(bad_all)
        final_pd[snr] = len(good_all)/total if total > 0 else np.nan
        final_rmse[snr] = np.sqrt(np.mean(good_all**2)) if len(good_all) > 0 else np.nan
    return final_pd, final_rmse

def mean_pd(pd_dict):
    return float(np.nanmean(list(pd_dict.values())))

def subsample_per_snr(data, feat, meta, n_per_snr, block_size=1000, n_blocks=8):
    idx = np.concatenate([np.arange(i*block_size, i*block_size+n_per_snr) for i in range(n_blocks)])
    return data[idx], feat[idx], meta[idx]

SUB_DATA, SUB_FEAT, SUB_META = subsample_per_snr(EVAL_DATA, EVAL_FEAT, EVAL_META, N_PER_SNR_EVAL)
print(f'✅ Evaluation ready ({SUB_DATA.shape[0]} samples for final comparison)')

In [ ]:
# Cell 12b -- MEASURE real eval throughput before trusting any time budget (don't guess).
# evaluate_on_bank's cost is dominated by per-sample CPU blob detection + Hungarian
# matching, not GPU inference -- it does NOT scale the way training does, so
# EVAL_RESERVE_MINUTES in Cell 5 was a guess until this cell runs it for real.
# Benchmarks every model Cell 15 will actually call: all 4 archs + the teacher.
# Inference cost only depends on architecture + output shape, not on trained vs
# random-init weights, so timing this now (before training) is valid.

BENCH_N = min(16, SUB_DATA.shape[0])
bench_models = list(models.items())
if teacher is not None:
    bench_models = bench_models + [('Teacher', teacher)]

per_model_sec_per_sample = {}
t_bench0 = time.time()
for name, model in bench_models:
    t0 = time.time()
    evaluate_on_bank(model, SUB_DATA[:BENCH_N], SUB_FEAT[:BENCH_N], SUB_META[:BENCH_N])
    dt = time.time() - t0
    per_model_sec_per_sample[name] = dt / BENCH_N
    print(f'  [{name}] measured: {dt:.1f}s for {BENCH_N} samples -> {dt/BENCH_N*1000:.0f} ms/sample')

bench_cost_min = (time.time() - t_bench0) / 60
print(f'Benchmark itself cost {bench_cost_min:.1f} min (comes out of the eval reserve below)')

n_full = N_PER_SNR_EVAL * 8
total_sec_per_sample = sum(per_model_sec_per_sample.values())
projected_minutes = total_sec_per_sample * n_full / 60
print(f'Projected Cell 15 eval time at N_PER_SNR_EVAL={N_PER_SNR_EVAL} '
      f'({n_full} samples x {len(bench_models)} models): {projected_minutes:.1f} min')

available_minutes = EVAL_RESERVE_MINUTES - bench_cost_min
if projected_minutes > available_minutes:
    safety = 0.85
    fit_samples = max(8 * len(bench_models), int(available_minutes * 60 * safety / total_sec_per_sample))
    N_PER_SNR_EVAL_NEW = max(4, fit_samples // 8)
    print(f'MEASURED throughput will NOT fit the {EVAL_RESERVE_MINUTES} min reserve as configured '
          f'(would need ~{projected_minutes:.0f} min).')
    print(f'Auto-shrinking N_PER_SNR_EVAL: {N_PER_SNR_EVAL} -> {N_PER_SNR_EVAL_NEW}')
    N_PER_SNR_EVAL = N_PER_SNR_EVAL_NEW
    SUB_DATA, SUB_FEAT, SUB_META = subsample_per_snr(EVAL_DATA, EVAL_FEAT, EVAL_META, N_PER_SNR_EVAL)
    print(f'New eval set: {SUB_DATA.shape[0]} samples ({N_PER_SNR_EVAL}/SNR x 8 SNRs)')
else:
    print(f'MEASURED throughput fits the {EVAL_RESERVE_MINUTES} min reserve '
          f'({projected_minutes:.1f} min projected, {available_minutes:.1f} min available).')

In [ ]:
def try_batch(body_fn, batch):
    try:
        probe = build_model_with_body(body_fn, name='probe')
        opt = tf.keras.optimizers.Adam(1e-3)
        x = tf.random.normal((batch, 64, 64, 2)); y = tf.random.normal((batch, M, M, 1))
        with tf.GradientTape() as tape:
            pred = probe(x, training=True)
            loss = tf.reduce_mean(tf.square(pred - y))
        grads = tape.gradient(loss, probe.trainable_variables)
        opt.apply_gradients(zip(grads, probe.trainable_variables))
        del probe, opt
        return True
    except tf.errors.ResourceExhaustedError:
        return False

def make_train_ds(batch):
    def fn():
        for d, g in training_data_generator(sigma=SIGMA, M=M):
            yield d, g
    ds = tf.data.Dataset.from_generator(
        fn, output_signature=(tf.TensorSpec(shape=(64,64,2), dtype=tf.float32),
                              tf.TensorSpec(shape=(M,M,1), dtype=tf.float32)))
    return ds.batch(batch).prefetch(tf.data.AUTOTUNE)

def train_architecture(name, model, body_fn, budget_hours):
    batch = 8
    for candidate in [16, 8]:
        if try_batch(body_fn, candidate):
            batch = candidate; break
    opt = tf.keras.optimizers.Adam(1e-3)   # fresh optimizer for the real model, untouched by the probe

    @tf.function
    def train_step(x, y):
        with tf.GradientTape() as tape:
            pred = model(x, training=True)
            loss = tf.reduce_mean(tf.square(pred - y))
        grads = tape.gradient(loss, model.trainable_variables)
        opt.apply_gradients(zip(grads, model.trainable_variables))
        return loss

    ds_iter = iter(make_train_ds(batch))
    t0 = time.time(); history = []; epoch = 0
    while (time.time() - t0) / 3600 < budget_hours:
        epoch_losses = []
        for step in range(STEPS_PER_EPOCH):
            x, y = next(ds_iter)
            epoch_losses.append(float(train_step(x, y)))
            if (time.time() - t0) / 3600 >= budget_hours: break
        history.append(float(np.mean(epoch_losses)))
        epoch += 1
        print(f'  [{name}] epoch {epoch}: loss={history[-1]:.5f}  ({(time.time()-t0)/60:.1f}min, batch={batch})')
    return history, batch

print('✅ Training loop ready (probe uses a throwaway model, real model starts clean)')

In [ ]:
histories = {}
batches_used = {}
train_ceiling_hours = TOTAL_TIME_BUDGET_HOURS - EVAL_RESERVE_MINUTES/60

for name, body_fn in ARCHS.items():
    model = models[name]
    if elapsed_hours() + PER_ARCH_BUDGET_HOURS > train_ceiling_hours:
        print(f'Training-time ceiling reached (eval reserve protected) -- skipping remaining architectures at {name}')
        break
    print()
    print('=== Training', name, '(budget:', round(PER_ARCH_BUDGET_HOURS*60, 1), 'min) ===')
    hist, batch = train_architecture(name, model, body_fn, PER_ARCH_BUDGET_HOURS)
    histories[name] = hist; batches_used[name] = batch

print()
print('All training done. Elapsed:', round(elapsed_hours(), 2), 'h  (eval reserve:', EVAL_RESERVE_MINUTES, 'min protected)')

plt.figure(figsize=(8,4))
for name, hist in histories.items():
    plt.plot(hist, label=name)
plt.xlabel('Epoch'); plt.ylabel('Loss'); plt.title('Training loss, all 4 architectures (equal budget)')
plt.legend(); plt.grid(alpha=.3); plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, 'screening_loss_curves.png'), dpi=130)
plt.show()

In [ ]:
# Cell 15 — Final comparison: all trained architectures vs teacher, on the same eval subsample
results = {}
for name, model in models.items():
    if name not in histories:
        continue   # wasn't trained (budget ran out)
    pd_, rmse_ = evaluate_on_bank(model, SUB_DATA, SUB_FEAT, SUB_META, batch_size=batches_used.get(name, 8))
    results[name] = {'pd': pd_, 'rmse': rmse_, 'mean_pd': mean_pd(pd_), 'params': model.count_params()}
    print(f'{name:>16}: mean Pd = {results[name]["mean_pd"]:.4f}  ({model.count_params():,} params)')

if teacher is not None:
    t_pd, t_rmse = evaluate_on_bank(teacher, SUB_DATA, SUB_FEAT, SUB_META)
    results['Teacher(ref,64blk)'] = {'pd': t_pd, 'rmse': t_rmse, 'mean_pd': mean_pd(t_pd), 'params': teacher.count_params()}
    print(f'{"Teacher(ref,64blk)":>16}: mean Pd = {results["Teacher(ref,64blk)"]["mean_pd"]:.4f}  '
          f'({teacher.count_params():,} params)  <- NOT equal-depth, reference only')

print('\n' + '='*60)
print('SCREENING VERDICT (8-block, ~20min budget each, mean Pd across SNR)')
print('='*60)
ranked = sorted([(k,v) for k,v in results.items() if 'Teacher' not in k], key=lambda kv: -kv[1]['mean_pd'])
for i, (name, r) in enumerate(ranked, 1):
    print(f'{i}. {name:<18} mean Pd = {r["mean_pd"]:.4f}')
print('\n(Compare against the 8-block trend, not directly against the 64-block pretrained teacher --')
print(' the point is RELATIVE promise among the 4 candidates at equal, fair footing.)')

In [ ]:
# Cell 16 — Save results
out = {
    'n_blocks': N_BLOCKS, 'per_arch_budget_min': PER_ARCH_BUDGET_HOURS*60,
    'n_per_snr_eval': N_PER_SNR_EVAL, 'total_elapsed_hours': elapsed_hours(),
    'results': {k: {'mean_pd': v['mean_pd'], 'params': int(v['params']),
                    'pd': {str(s): float(p) for s,p in v['pd'].items()},
                    'rmse': {str(s): float(r) for s,r in v['rmse'].items()}}
               for k, v in results.items()},
    'batches_used': batches_used,
}
with open(os.path.join(OUT_DIR, 'architecture_screening_results.json'), 'w') as f:
    json.dump(out, f, indent=2)
print(f'Saved: {os.path.join(OUT_DIR, "architecture_screening_results.json")}')
print('\n⚠️ Remember: Save Version -> Save & Run All (Commit) so this persists.')